# **Configurations**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/College/FLEX-Med/backend/federated_learning

# Dependency Installation
!pip install fastapi uvicorn pyngrok nest-asyncio flwr["simulation"] flwr_datasets datasets supabase -q

PROJECT_ROOT = "/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning


# **Verify if the environment is setup for FL simulation**

In [ ]:
#!/usr/bin/env python3
"""
FL Health Verification Script (Jupyter / Colab Safe)

This version is safe to run inside:
- Jupyter Notebook
- Google Colab
- Local terminal (python verify_fl_health.py)

It avoids __file__ and resolves paths dynamically.
"""

import os
import json
import sys
from pathlib import Path
from typing import Dict, List, Tuple

# =====================================================
# PROJECT ROOT RESOLUTION (JUPYTER SAFE)
# =====================================================
# Assumes notebook is launched from flex-med root
PROJECT_ROOT = Path.cwd()

# If flex_med is not importable, auto-fix sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# =====================================================
# ANSI COLORS
# =====================================================
class Colors:
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BLUE = '\033[94m'
    BOLD = '\033[1m'
    END = '\033[0m'


def print_header(text: str):
    print(f"\n{Colors.BOLD}{Colors.BLUE}{'=' * 70}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.BLUE}{text.center(70)}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.BLUE}{'=' * 70}{Colors.END}\n")


def print_success(text: str):
    print(f"{Colors.GREEN}✓ {text}{Colors.END}")


def print_warning(text: str):
    print(f"{Colors.YELLOW}⚠ {text}{Colors.END}")


def print_error(text: str):
    print(f"{Colors.RED}✗ {text}{Colors.END}")


def print_info(text: str):
    print(f"  {text}")


# =====================================================
# DATASET CHECK
# =====================================================
def check_dataset_structure(
    dataset_path: Path,
    expected_classes: List[str] = ["all", "hem"]
) -> Tuple[bool, Dict]:

    stats = {}

    if not dataset_path.exists():
        return False, {"error": "Dataset path does not exist"}

    for class_name in expected_classes:
        class_path = dataset_path / class_name
        if not class_path.exists():
            return False, {"error": f"Missing class directory: {class_name}"}

        image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        images = [f for f in class_path.iterdir() if f.suffix.lower() in image_exts]
        stats[class_name] = len(images)

    total = sum(stats.values())
    if total == 0:
        return False, {"error": "No images found in dataset"}

    for cls in expected_classes:
        stats[f"{cls}_pct"] = stats[cls] / total * 100

    stats["total"] = total
    return True, stats


def verify_datasets(use_colab: bool = True) -> bool:
    print_header("Step 1: Dataset Integrity Check")

    if use_colab:
        base_path = Path("/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc")
    else:
        base_path = PROJECT_ROOT / "cnmc_datasets"

    datasets = {
        "Client 0": base_path / "cnmc_client_0",
        "Client 1": base_path / "cnmc_client_1",
        "Public Anchor": base_path / "cnmc_public_anchor",
        "Public Test": base_path / "cnmc_public_test",
    }

    all_valid = True

    for name, path in datasets.items():
        valid, stats = check_dataset_structure(path)
        if valid:
            print_success(f"{name}: {path}")
            print_info(f"Total images: {stats['total']}")
            print_info(f"  Leukemia (all): {stats['all']} ({stats['all_pct']:.1f}%)")
            print_info(f"  Healthy (hem):  {stats['hem']} ({stats['hem_pct']:.1f}%)")
        else:
            print_error(f"{name}: {path}")
            print_info(f"  Error: {stats.get('error')}")
            all_valid = False

    return all_valid


# =====================================================
# CLIENT CONFIG CHECK
# =====================================================
def verify_client_config() -> bool:
    print_header("Step 2: Client Configuration Check")

    config_path = PROJECT_ROOT / "flex_med" / "utils" / "client_data.json"

    if not config_path.exists():
        print_error(f"Configuration file not found: {config_path}")
        return False

    # --- FIX STARTS HERE ---
    # Check if file is empty before trying to load
    if config_path.stat().st_size == 0:
        print_error(f"Config file is empty (0 bytes): {config_path}")
        print_info("Action needed: Re-generate client_data.json")
        return False

    try:
        with open(config_path, 'r') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        print_error(f"Corrupted JSON in {config_path}")
        print_info(f"Error details: {e}")
        return False
    # --- FIX ENDS HERE ---

    clients = data.get('clients', [])

    print_success(f"Loaded config: {config_path}")
    print_info(f"Number of clients: {len(clients)}")

    if not clients:
        print_error("No clients found in 'clients' key.")
        return False

    for client in clients:
        c_id = client.get('id', 'Unknown')
        c_name = client.get('client_name', 'Unknown')
        c_model = client.get('model_type', 'Unknown')

        print_info(f"\n  Client {c_id}: {c_name}")
        print_info(f"    Model: {c_model}")

        model_path = Path(client['model_path'])
        if model_path.exists():
            print_warning("    Existing model checkpoint found")
        else:
            print_success("    Fresh training (no checkpoint)")

    return True


# =====================================================
# FL CONFIG CHECK
# =====================================================
def verify_fl_config() -> bool:
    print_header("Step 3: FL Configuration Parameters")

    config_path = PROJECT_ROOT / "pyproject.toml"

    if not config_path.exists():
        print_error("pyproject.toml not found")
        return False

    content = config_path.read_text()
    params = {}

    for line in content.splitlines():
        if '=' in line and not line.strip().startswith('#'):
            k, v = line.split('=', 1)
            params[k.strip()] = v.split('#')[0].strip()

    print_success("FL config loaded")
    print_info(f"Rounds: {params.get('num-server-rounds')}")
    print_info(f"Local epochs: {params.get('local-epochs')}")
    print_info(f"Batch size: {params.get('batch-size')}")
    print_info(f"LR: {params.get('lr')}")

    lr_decay = float(params.get('lr-decay', 0))
    if lr_decay == 0.98:
        print_success("LR decay = 0.98 ✓")
    else:
        print_warning(f"LR decay = {lr_decay} (expected 0.98)")

    return True

# =====================================================
# CLEAN STATE
# =====================================================
def verify_clean_state(use_colab: bool = True) -> bool:
    print_header("Step 5: Clean State Verification")

    models_dir = (
        Path("/content/drive/MyDrive/College/FLEX-Med/backend/models")
        if use_colab
        else PROJECT_ROOT / "models"
    )

    if not models_dir.exists():
        print_success("No existing models directory (fresh)")
        return True

    checkpoints = list(models_dir.glob("*.pt"))
    if checkpoints:
        print_warning(f"Found {len(checkpoints)} checkpoint(s)")
    else:
        print_success("No checkpoints found")

    return True


# =====================================================
# MAIN
# =====================================================
def main():
    print(f"\n{Colors.BOLD}FL Health Verification Tool (Notebook Mode){Colors.END}")

    results = [
        verify_datasets(use_colab=True),
        verify_client_config(),
        verify_fl_config(),
        verify_clean_state(use_colab=True),
    ]

    print_header("Verification Summary")

    if all(results):
        print_success("All checks passed — Ready for FL simulation 🚀")
    else:
        print_error("Some checks failed — Fix issues before training")


# Run immediately in notebook
main()


FL Health Verification Tool (Notebook Mode)

                   Step 1: Dataset Integrity Check                    

✓ Client 0: /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_client_0
  Total images: 2561
    Leukemia (all): 2455 (95.9%)
    Healthy (hem):  106 (4.1%)
✓ Client 1: /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_client_1
  Total images: 6213
    Leukemia (all): 4160 (67.0%)
    Healthy (hem):  2053 (33.0%)
✓ Public Anchor: /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_anchor
  Total images: 2504
    Leukemia (all): 1252 (50.0%)
    Healthy (hem):  1252 (50.0%)
✓ Public Test: /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_test
  Total images: 1252
    Leukemia (all): 626 (50.0%)
    Healthy (hem):  626 (50.0%)

                  Step 2: Client Configuration Check                  

✓ Loaded config: /content/drive/MyDrive/College/FLEX-Med/backend/federated_learning/flex_med/util

# **API Definition**

In [ ]:
import os
import sys
import json
import logging
import subprocess
from typing import List, Tuple
import re
from datetime import datetime

from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel
from starlette.middleware.cors import CORSMiddleware

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================

# Import configuration from config.py
try:
    from flex_med.utils.config import (
        PROJECT_DIR,
        SIMULATION_LOG_PATH,
        CLIENT_INFO_FILE_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
    )
except ImportError:
    import sys
    sys.path.append(os.getcwd())
    from flex_med.utils.config import (
        PROJECT_DIR,
        SIMULATION_LOG_PATH,
        CLIENT_INFO_FILE_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
    )

os.makedirs(PROJECT_DIR, exist_ok=True)

# ==========================================
# 2. SERVER LOGGING SETUP
# ==========================================

logger = logging.getLogger("inference_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

stream_handler = logging.StreamHandler(sys.stdout)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
stream_handler.setFormatter(formatter)

logger.addHandler(stream_handler)

logger.info("=== Flex-Med Orchestrator Started ===")

# ==========================================
# 3. CONFIGURATION VALIDATION
# ==========================================

def validate_client_config(client) -> Tuple[bool, str]:
    """
    Validate client configuration

    Returns:
        (is_valid, error_message)
    """
    valid_models = [
        'resnet50',         # The Industry Standard
        'mobilenet_v2',     # Mobile Optimized
        'densenet121',      # High Dense Connections
        'efficientnet_b0',  # Modern Efficiency Optimizer
    ]
    if client.model_type not in valid_models:
        return False, f"Invalid model type: {client.model_type}. Must be one of {valid_models}"

    if not client.model_path:
        return False, f"Client {client.client_name} has no model_path"

    model_dir = os.path.dirname(client.model_path)
    if not os.path.exists(model_dir):
        try:
            os.makedirs(model_dir, exist_ok=True)
            logger.info(f"Created model directory: {model_dir}")
        except Exception as e:
            return False, f"Cannot create model directory: {model_dir}, error: {e}"

    return True, ""

# ==========================================
# 4. CREATE RUN SCRIPT
# ==========================================

def create_run_script(simulation_id: int, supabase_url: str, supabase_key: str):
    """
    Creates the shell script that runs the Flower simulation.

    Args:
        simulation_id: ID of the simulation in fl_simulations table
        supabase_url: Supabase project URL
        supabase_key: Supabase API key
    """
    script_content = f"""#!/bin/bash
cd "{PROJECT_DIR}"

echo "[Script] Starting Flex-Med Simulation at $(date)..."
echo "[Script] Simulation ID: {simulation_id}"

# Export environment variables for FL system
export FLEX_MED_SIMULATION_ID="{simulation_id}"
export SUPABASE_URL="{supabase_url}"
export SUPABASE_KEY="{supabase_key}"

# Optional: Configure Ray memory settings
export RAY_memory_usage_threshold="0.98"

# Only install the local package in editable mode (Fast & Lightweight)
pip install -e . -q

# Run Flower CLI directly
flwr run .
"""

    with open(RUN_SIMULATION_SHELL_FILE_PATH, "w") as f:
        f.write(script_content)

    os.chmod(RUN_SIMULATION_SHELL_FILE_PATH, 0o755)
    logger.info(f"Created run_simulation.sh at {RUN_SIMULATION_SHELL_FILE_PATH}")

    return RUN_SIMULATION_SHELL_FILE_PATH

# ==========================================
# 5. BACKGROUND TASK
# ==========================================

def run_simulation_task():
    """Runs the FL simulation in background."""
    logger.info("="*70)
    logger.info("Starting Flower Simulation in Background")
    logger.info("="*70)

    try:
        logger.info("\n[Background Task] Running Flower Simulation...")

        with open(SIMULATION_LOG_PATH, "w") as log_file:
            process = subprocess.run(
                ["bash", SCRIPT_PATH],
                stdout=log_file,
                stderr=log_file,
                text=True
            )

        return_code = process.returncode

        if return_code == 0:
            logger.info("\n" + "="*70)
            logger.info("✅ Federated Learning Simulation COMPLETED Successfully")
            logger.info("="*70)

            completion_status = {
                "status": "completed",
                "timestamp": datetime.now().isoformat(),
                "return_code": return_code,
                "log_path": SIMULATION_LOG_PATH
            }

        else:
            logger.error("\n" + "="*70)
            logger.error(f"❌ Federated Learning Simulation FAILED")
            logger.error(f"   Return Code: {return_code}")
            logger.error("="*70)

            error_status = {
                "status": "failed",
                "timestamp": datetime.now().isoformat(),
                "return_code": return_code,
                "error": "Simulation failed",
                "log_path": SIMULATION_LOG_PATH
            }

    except Exception as e:
        logger.error(f"\n❌ Simulation error: {e}")
        import traceback
        logger.error(traceback.format_exc())

        try:
            error_status = {
                "status": "exception",
                "timestamp": datetime.now().isoformat(),
                "error": str(e),
                "traceback": traceback.format_exc(),
                "log_path": SIMULATION_LOG_PATH
            }

        except:
            pass

# ==========================================
# 6. PYDANTIC MODELS (v3.0 - Removed deprecated fields)
# ==========================================

class ClientConfig(BaseModel):
    """
    Client configuration schema for v3.0
    
    Removed fields (deprecated):
    - has_local_data (all clients now use Dirichlet partitioning)
    - dataset_path (replaced by shared LOCAL_TRAIN_DATASET_PATH)
    """
    id: int
    created_at: str
    client_name: str
    model_type: str
    status: str
    model_path: str
    metrics: str

class StartFLRequest(BaseModel):
    """
    New request format for FL simulation with database integration.

    Fields:
        simulation_id: ID of the simulation record in fl_simulations table
        clients: List of client configurations
        supabase_url: Supabase project URL for direct database access
        supabase_key: Supabase API key (anon key)
    """
    simulation_id: int
    clients: List[ClientConfig]
    supabase_url: str
    supabase_key: str

# ==========================================
# 7. FASTAPI APP
# ==========================================

app = FastAPI(title="Flex-Med Orchestrator")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# 8. ENDPOINTS
# ==========================================

@app.get("/")
def read_root():
    return {"message": "Flex-Med Orchestrator is running"}

def update_data_json(simulation_id: int, clients: List[ClientConfig], path: str = CLIENT_INFO_FILE_PATH):
    """
    Save client configuration to data.json with simulation metadata.

    Args:
        simulation_id: Simulation ID for tracking
        clients: List of client configurations
        path: Path to save the configuration file
    """
    try:
        client_dicts = [c.dict() for c in clients]

        # Wrap clients with simulation metadata (new format)
        config_data = {
            "simulation_id": simulation_id,
            "clients": client_dicts
        }

        os.makedirs(os.path.dirname(path), exist_ok=True)

        with open(path, "w") as f:
            json.dump(config_data, f, indent=2)

        logger.info(f"✓ Updated data.json with simulation_id={simulation_id} and {len(clients)} clients")
        return True
    except Exception as e:
        logger.error(f"Failed to update data.json: {e}")
        return False

def update_pyproject_toml(num_clients: int, path: str = PYPROJECT_PATH):
    """Update pyproject.toml with correct num-supernodes"""
    try:
        if not os.path.exists(path):
            logger.error(f"pyproject.toml not found at {path}")
            return False

        with open(path, 'r') as f:
            content = f.read()

        # Fixed regex: Match 'num-supernodes = N' directly (not 'options.num-supernodes')
        pattern = r'(num-supernodes\s*=\s*)\d+'

        if not re.search(pattern, content):
            logger.warning("num-supernodes not found in pyproject.toml")
            return False

        # Count how many occurrences will be updated
        matches = re.findall(pattern, content)
        old_match = re.search(r'num-supernodes\s*=\s*(\d+)', content)
        old_value = old_match.group(1) if old_match else "unknown"

        # Replace ALL occurrences (both local-simulation and app.federations sections)
        new_content = re.sub(pattern, f'\\g<1>{num_clients}', content)

        with open(path, 'w') as f:
            f.write(new_content)

        logger.info(f"✓ Updated pyproject.toml: num-supernodes {old_value} → {num_clients} ({len(matches)} occurrences)")
        return True

    except Exception as e:
        logger.error(f"Failed to update pyproject.toml: {e}")
        return False

@app.post("/start_fl")
async def start_fl(
    request: StartFLRequest,
    background_tasks: BackgroundTasks,
):
    """
    Start federated learning simulation with v3.0 schema (no deprecated fields).

    This endpoint now receives:
    - simulation_id: ID of the simulation record in fl_simulations table
    - clients: List of client configurations (without has_local_data/dataset_path)
    - supabase_url: Supabase URL for direct database access
    - supabase_key: Supabase API key

    All clients now use runtime Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH.

    Steps:
    1. Validates client configurations
    2. Saves client config to data.json (with simulation_id wrapper)
    3. Updates pyproject.toml with correct num-supernodes
    4. Creates run script with environment variables
    5. Triggers the FL simulation in background
    """

    simulation_id = request.simulation_id
    clients = request.clients
    num_clients = len(clients)

    logger.info(f"=" * 70)
    logger.info(f"Received FL start request")
    logger.info(f"  - Simulation ID: {simulation_id}")
    logger.info(f"  - Number of clients: {num_clients}")
    logger.info(f"=" * 70)

    # Log client details (v3.0: removed has_local_data status)
    for i, client in enumerate(clients):
        logger.info(f"  Client {i}: {client.client_name:20} | {client.model_type:15}")

    # ===== VALIDATION: CHECK CLIENT CONFIGS =====
    logger.info("\n[Endpoint] Validating client configurations...")

    validation_errors = []
    for i, client in enumerate(clients):
        is_valid, error_msg = validate_client_config(client)
        if not is_valid:
            validation_errors.append(f"Client {i}: {error_msg}")

    if validation_errors:
        error_detail = "\n".join(validation_errors)
        logger.error(f"Configuration validation failed:\n{error_detail}")
        raise HTTPException(
            status_code=400,
            detail=f"Invalid client configuration:\n{error_detail}"
        )

    logger.info("✓ All client configurations validated")

    # ===== STEP 1: UPDATE DATA.JSON (with simulation_id) =====
    logger.info("\n[Endpoint] Step 1: Updating data.json...")

    if not update_data_json(simulation_id, clients, CLIENT_INFO_FILE_PATH):
        raise HTTPException(
            status_code=500,
            detail="Failed to save client configuration to data.json"
        )

    # ===== STEP 2: UPDATE PYPROJECT.TOML =====
    logger.info("\n[Endpoint] Step 2: Updating pyproject.toml...")

    if not update_pyproject_toml(num_clients, PYPROJECT_PATH):
        logger.warning("Failed to update pyproject.toml, but continuing...")

    # ===== STEP 3: CREATE RUN SCRIPT (with environment variables) =====
    logger.info("\n[Endpoint] Step 3: Creating run script with environment variables...")
    logger.info(f"  - FLEX_MED_SIMULATION_ID={simulation_id}")
    logger.info(f"  - SUPABASE_URL={request.supabase_url}")
    logger.info(f"  - SUPABASE_KEY=***hidden***")

    global SCRIPT_PATH
    SCRIPT_PATH = create_run_script(
        simulation_id=simulation_id,
        supabase_url=request.supabase_url,
        supabase_key=request.supabase_key
    )

    # ===== STEP 4: SET ENVIRONMENT VARIABLES =====
    # Also set environment variables for the current process
    # (in case subprocess doesn't inherit them properly)
    os.environ['FLEX_MED_SIMULATION_ID'] = str(simulation_id)
    os.environ['SUPABASE_URL'] = request.supabase_url
    os.environ['SUPABASE_KEY'] = request.supabase_key

    # Set client configs as JSON string (alternative to data.json)
    client_config_json = json.dumps({
        "simulation_id": simulation_id,
        "clients": [c.dict() for c in clients]
    })
    os.environ['FLEX_MED_CLIENT_CONFIGS'] = client_config_json

    # ===== STEP 5: TRIGGER SIMULATION =====
    logger.info("\n[Endpoint] Step 4: Starting FL simulation...")
    logger.info(f"  - Number of clients: {num_clients}")
    logger.info(f"  - All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")
    logger.info(f"  - Database: Metrics will be saved to client_simulation_metrics table")
    logger.info(f"=" * 70 + "\n")

    # Trigger the background task
    background_tasks.add_task(run_simulation_task)

    return {
        "status": "accepted",
        "message": f"Federated simulation started with {num_clients} clients",
        "simulation_id": simulation_id,
        "num_clients": num_clients,
        "config_updated": {
            "data_json": CLIENT_INFO_FILE_PATH,
            "pyproject_toml": PYPROJECT_PATH
        },
        "log_path": SIMULATION_LOG_PATH,
        "database": {
            "metrics_table": "client_simulation_metrics",
            "simulation_table": "fl_simulations"
        },
        "version": "3.0"
    }

print("✅ Flex-Med Orchestrator Ready (v3.0)")
print("📋 Endpoint: POST /start_fl")
print("📋 Schema: {simulation_id, clients, supabase_url, supabase_key}")
print("📋 Note: has_local_data and dataset_path fields removed")
print("📋 All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")

# **NGROK Setup**

In [7]:
import nest_asyncio
import uvicorn
import subprocess
import threading
from pyngrok import ngrok
from pyngrok.process import kill_process
from fastapi import FastAPI, BackgroundTasks, HTTPException
from starlette.middleware.cors import CORSMiddleware

# 0. Cleanup
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

# 1. Auth & Tunnel
# REPLACE WITH YOUR ACTUAL TOKEN
AUTH_TOKEN = "38BI51ntfzceNwdU1BTYDLnAZib_3S3SmcwQJFykk89BGBmEJ"
ngrok.set_auth_token(AUTH_TOKEN)

http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print(f"🌐 API URL: {public_url}")
print(f"🚀 POST to start FL: {public_url}/start_fl")

# 2. Run Uvicorn
nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, port=8000, host="0.0.0.0")

thread = threading.Thread(target=run_uvicorn)
thread.start()

🌐 API URL: https://intraspinal-agape-deidra.ngrok-free.dev
🚀 POST to start FL: https://intraspinal-agape-deidra.ngrok-free.dev/start_fl


# **View Training Logs**

In [ ]:
!tail -f "/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning/flex_med/utils/simulation_run.log"

[Script] Starting Flex-Med Simulation at Wed Jan 28 05:02:54 PM UTC 2026...
[Script] Simulation ID: 10
Loading project configuration... 
Success
2026-01-28 17:03:06.482653: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769619786.506007   13656 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769619786.512516   13656 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769619786.532139   13656 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769619786.532480   13656 computation_placer.cc:177] computation placer already registered. Please check 

INFO:inference_logger:


2026-01-28 17:29:50,325 - INFO - ✅ Federated Learning Simulation COMPLETED Successfully


INFO:inference_logger:✅ Federated Learning Simulation COMPLETED Successfully


2026-01-28 17:29:50,328 - INFO - ======================================================================


INFO:inference_logger:======================================================================


# **Kill process**

In [ ]:
# Kill any process running on port 8000
!fuser -k 8000/tcp

print("Attempted to kill processes on port 8000. Please re-run the FastAPI server cell now.")

# **Model Evaluation & Database Metrics Visualization**

In [ ]:
# ==========================================
# FL METRICS VISUALIZATION (WITH PRE-FL BASELINE)
# ==========================================

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from supabase import create_client, Client
from typing import Optional, Dict, List

# ==========================================
# CONFIGURATION
# ==========================================

MODEL_PATH = "/content/drive/MyDrive/College/FLEX-Med/backend/models/Delmon.pt"
SUPABASE_URL = os.getenv("SUPABASE_URL", "YOUR_SUPABASE_URL_HERE")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "YOUR_SUPABASE_ANON_KEY_HERE")
SIMULATION_ID: Optional[int] = None  # None = latest simulation
CLIENT_ID: Optional[int] = None  # None = all clients, or specific client ID (e.g., 1, 2)

# ==========================================
# HELPER FUNCTIONS
# ==========================================

def connect_to_supabase() -> Client:
    """Initialize Supabase client."""
    if SUPABASE_URL == "YOUR_SUPABASE_URL_HERE":
        raise ValueError("⚠️ Please set SUPABASE_URL and SUPABASE_KEY")
    print(f"🔌 Connecting to Supabase...")
    supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
    print(f"✅ Connected\n")
    return supabase

def fetch_metrics(supabase: Client, simulation_id: Optional[int] = None, client_id: Optional[int] = None):
    """Fetch and parse metrics from database.
    
    This function extracts ROUND-BY-ROUND metrics to show training progress:
    - Round 0: Pre-FL baseline (before training)
    - Round 1+: Validation metrics after each FL round
    
    Args:
        supabase: Supabase client
        simulation_id: Simulation ID (None = latest)
        client_id: Client ID to filter by (None = all clients)
    """
    print(f"📊 Fetching metrics...")
    
    # Get latest simulation if not specified
    if simulation_id is None:
        response = supabase.table('fl_simulations').select('id').order('created_at', desc=True).limit(1).execute()
        if not response.data:
            raise ValueError("No simulations found")
        simulation_id = response.data[0]['id']
        print(f"   Using simulation ID: {simulation_id}")
    
    # Fetch metrics with optional client filter
    query = supabase.table('client_simulation_metrics') \
        .select('*') \
        .eq('simulation_id', simulation_id)
    
    if client_id is not None:
        query = query.eq('client_id', client_id)
        print(f"   Filtering by client_id: {client_id}")
    
    response = query.execute()
    
    if not response.data:
        if client_id is not None:
            raise ValueError(f"No metrics found for simulation_id={simulation_id}, client_id={client_id}")
        else:
            raise ValueError(f"No metrics found for simulation_id={simulation_id}")
    
    print(f"✅ Fetched {len(response.data)} client record(s)\n")
    
    # Parse into structured format
    clients_data = []
    for record in response.data:
        metrics = json.loads(record['metrics']) if isinstance(record['metrics'], str) else record['metrics']
        
        # ==========================================
        # EXTRACT ROUND-BY-ROUND METRICS
        # ==========================================
        
        # Round 0: PRE-FL baseline (before any training)
        pre_fl = metrics.get('global', {}).get('pre_fl', {})
        rounds = [{
            'round': 0,
            'accuracy': pre_fl.get('accuracy', 0) * 100,
            'loss': pre_fl.get('loss', 0),
            'f1_score': pre_fl.get('f1_score', 0) * 100,
            'recall': pre_fl.get('recall', 0) * 100,
            'precision': pre_fl.get('precision', 0) * 100,
        }]
        
        # Round 1+: Per-round validation metrics (ACTUAL TRAINING PROGRESS)
        for round_entry in metrics.get('rounds', []):
            val = round_entry.get('validation', {})
            rounds.append({
                'round': round_entry['round'],
                'accuracy': val.get('accuracy', 0) * 100,
                'loss': val.get('loss', 0),
                'f1_score': val.get('f1_score', 0) * 100,
                'recall': val.get('recall', 0) * 100,
                'precision': val.get('precision', 0) * 100,
            })
        
        # Use LAST ROUND metrics as final metrics (consistent with graphs)
        last_round = rounds[-1] if len(rounds) > 0 else rounds[0]
        
        # Extract confusion matrix from post-FL global evaluation
        post_fl = metrics.get('global', {}).get('post_fl', {})
        cm = post_fl.get('confusion_matrix', {})
        
        # Calculate improvement from Round 0 to Last Round
        pre_fl_accuracy = rounds[0]['accuracy']
        final_accuracy = last_round['accuracy']
        improvement = final_accuracy - pre_fl_accuracy
        
        clients_data.append({
            'client_id': record['client_id'],
            'client_name': record.get('client_name', f"Client {record['client_id']}"),
            'rounds': rounds,  # Full round-by-round progression
            'num_rounds': len(rounds) - 1,  # Excluding round 0
            'pre_fl_accuracy': pre_fl_accuracy,
            'final_accuracy': final_accuracy,  # From last round
            'final_loss': last_round['loss'],
            'final_f1': last_round['f1_score'],
            'final_recall': last_round['recall'],
            'final_precision': last_round['precision'],
            'improvement': improvement,
            'confusion_matrix': [[cm.get('TN', 0), cm.get('FP', 0)],
                                [cm.get('FN', 0), cm.get('TP', 0)]]
        })
    
    return clients_data

# ==========================================
# VISUALIZATION - INDIVIDUAL CLIENT GRAPHS
# ==========================================

def plot_individual_client_metrics(client_data: Dict, client_index: int = 0):
    """Plot all metrics for a single client showing ROUND-BY-ROUND progress.
    
    Args:
        client_data: Dictionary containing client metrics
        client_index: Index for color selection
    """
    
    color = plt.cm.tab10(client_index)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'{client_data["client_name"]} - Round-by-Round Training Progress\n(Round 0 = Pre-FL Baseline | Rounds 1-{client_data["num_rounds"]} = FL Training)', 
                 fontsize=16, fontweight='bold')
    
    metrics = ['accuracy', 'loss', 'f1_score', 'recall', 'precision']
    titles = ['Accuracy (%) per Round', 'Loss per Round', 'F1 Score (%) per Round', 
              'Recall (%) per Round', 'Precision (%) per Round']
    positions = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1)]
    
    for metric, title, pos in zip(metrics, titles, positions):
        ax = axes[pos]
        
        rounds = [r['round'] for r in client_data['rounds']]
        values = [r[metric] for r in client_data['rounds']]
        
        # Plot round-by-round progress
        ax.plot(rounds, values, marker='o', label=client_data['client_name'], 
               color=color, linewidth=3, markersize=8)
        
        # Highlight Round 0 (pre-FL baseline)
        ax.plot(0, values[0], marker='s', color=color, 
               markersize=12, markeredgewidth=2.5, markeredgecolor='black',
               linestyle='None', zorder=10, label='Pre-FL Baseline')
        
        # Highlight final round (post-FL)
        if len(rounds) > 1:
            ax.plot(rounds[-1], values[-1], marker='*', color='gold', 
                   markersize=15, markeredgewidth=2, markeredgecolor='black',
                   linestyle='None', zorder=10, label='Final Round')
        
        # Add value annotations for each round
        for i, (r, v) in enumerate(zip(rounds, values)):
            if metric != 'loss':  # For percentage metrics
                ax.annotate(f'{v:.1f}%', xy=(r, v), xytext=(0, 10),
                           textcoords='offset points', ha='center', fontsize=9,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
            else:  # For loss
                ax.annotate(f'{v:.3f}', xy=(r, v), xytext=(0, 10),
                           textcoords='offset points', ha='center', fontsize=9,
                           bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
        
        # Smart y-axis scaling to show growth clearly
        if len(values) > 1:
            min_val = min(values)
            max_val = max(values)
            range_val = max_val - min_val
            
            if metric == 'loss':
                # For loss, add padding and start from reasonable minimum
                padding = range_val * 0.2 if range_val > 0 else 0.1
                ax.set_ylim(max(0, min_val - padding), max_val + padding)
            else:
                # For accuracy/precision/recall/f1, zoom in on relevant range
                if range_val < 5:  # Small improvement - add fixed padding
                    padding = 5
                else:
                    padding = range_val * 0.2
                
                ax.set_ylim(max(0, min_val - padding), min(100, max_val + padding))
        
        ax.set_xlabel('Round (0=Pre-FL, 1+=Training)', fontweight='bold', fontsize=11)
        ax.set_ylabel(title, fontweight='bold', fontsize=11)
        ax.set_title(title, fontweight='bold', fontsize=12)
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Add vertical line to separate pre-FL from training
        if len(rounds) > 1:
            ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, linewidth=1.5)
        
        # Add improvement delta text
        if len(values) > 1:
            improvement = values[-1] - values[0]
            if metric != 'loss':
                improvement_text = f'Δ = +{improvement:.1f}%' if improvement >= 0 else f'Δ = {improvement:.1f}%'
                box_color = 'lightgreen' if improvement > 0 else 'lightcoral'
            else:
                improvement_text = f'Δ = {improvement:.3f}' if improvement < 0 else f'Δ = +{improvement:.3f}'
                box_color = 'lightgreen' if improvement < 0 else 'lightcoral'  # Lower loss is better
            
            ax.text(0.98, 0.02, improvement_text, transform=ax.transAxes,
                   fontsize=11, fontweight='bold', ha='right', va='bottom',
                   bbox=dict(boxstyle='round,pad=0.5', facecolor=box_color, alpha=0.7))
    
    # Hide the unused subplot
    axes[1, 2].axis('off')
    
    # Add summary text showing round-by-round progression
    rounds_summary = "\n".join([
        f"Round {r['round']}: Acc={r['accuracy']:.1f}%, Loss={r['loss']:.3f}"
        for r in client_data['rounds']
    ])
    
    summary_text = f"""Round-by-Round Summary:

{rounds_summary}

Total Improvement:
• Accuracy: +{client_data['improvement']:.2f}%
• From {client_data['pre_fl_accuracy']:.1f}% → {client_data['final_accuracy']:.1f}%

Final Metrics (Round {client_data['num_rounds']}):
• F1 Score: {client_data['final_f1']:.2f}%
• Recall: {client_data['final_recall']:.2f}%
• Precision: {client_data['final_precision']:.2f}%
• Loss: {client_data['final_loss']:.4f}
    """
    
    axes[1, 2].text(0.05, 0.95, summary_text, fontsize=10, verticalalignment='top',
                   family='monospace', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

def plot_all_clients_metrics(clients_data: List[Dict]):
    """Generate individual metric plots for each client."""
    
    print(f"📊 Generating individual round-by-round plots for {len(clients_data)} client(s)...\n")
    
    for idx, client in enumerate(clients_data):
        print(f"   Plotting {client['client_name']} ({client['num_rounds']} training rounds)...")
        plot_individual_client_metrics(client, idx)
    
    print(f"\n✅ Generated {len(clients_data)} individual client plot(s)")

def plot_improvement(clients_data: List[Dict]):
    """Bar chart showing pre-FL vs final accuracy improvement."""
    
    fig, ax = plt.subplots(figsize=(14, 7))
    
    client_names = [c['client_name'] for c in clients_data]
    x = np.arange(len(client_names))
    width = 0.35
    
    pre_fl_acc = [c['pre_fl_accuracy'] for c in clients_data]
    final_acc = [c['final_accuracy'] for c in clients_data]
    
    bars1 = ax.bar(x - width/2, pre_fl_acc, width, label='Pre-FL (Round 0)', 
                   color='#E63946', alpha=0.7)
    bars2 = ax.bar(x + width/2, final_acc, width, label=f'Final (Round {clients_data[0]["num_rounds"]})', 
                   color='#06D6A0', alpha=0.85)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for bar in bars2:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Add improvement arrows
    for idx, client in enumerate(clients_data):
        ax.annotate('', xy=(x[idx] + width/2, final_acc[idx] - 2), 
                   xytext=(x[idx] - width/2, pre_fl_acc[idx] + 2),
                   arrowprops=dict(arrowstyle='->', lw=2, color='#1D3557', alpha=0.6))
        
        # Improvement percentage
        improvement = client['improvement']
        ax.text(x[idx], (pre_fl_acc[idx] + final_acc[idx]) / 2, 
               f'+{improvement:.1f}%', ha='center', fontsize=11, 
               fontweight='bold', color='#1D3557',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    
    # Smart y-axis scaling
    all_values = pre_fl_acc + final_acc
    min_val = min(all_values)
    max_val = max(all_values)
    range_val = max_val - min_val
    padding = max(range_val * 0.2, 5)
    
    ax.set_ylim(max(0, min_val - padding), min(100, max_val + padding))
    
    ax.set_xlabel('Client', fontweight='bold', fontsize=12)
    ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
    ax.set_title('Round-by-Round Training: Pre-FL vs Final Accuracy', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(client_names)
    ax.legend(loc='upper left', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_final_comparison(clients_data: List[Dict]):
    """Bar chart comparing final metrics from last training round."""
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    client_names = [c['client_name'] for c in clients_data]
    x = np.arange(len(client_names))
    width = 0.15
    
    metrics = {
        'Accuracy': [c['final_accuracy'] for c in clients_data],
        'F1 Score': [c['final_f1'] for c in clients_data],
        'Recall': [c['final_recall'] for c in clients_data],
        'Precision': [c['final_precision'] for c in clients_data],
    }
    
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
    
    for idx, (label, values) in enumerate(metrics.items()):
        offset = (idx - 1.5) * width
        bars = ax.bar(x + offset, values, width, label=label, color=colors[idx], alpha=0.85)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1f}%', ha='center', va='bottom', fontsize=8)
    
    # Smart y-axis scaling
    all_values = [v for values in metrics.values() for v in values]
    min_val = min(all_values)
    max_val = max(all_values)
    range_val = max_val - min_val
    padding = max(range_val * 0.2, 5)
    
    ax.set_ylim(max(0, min_val - padding), min(100, max_val + padding))
    
    ax.set_xlabel('Client', fontweight='bold', fontsize=12)
    ax.set_ylabel('Score (%)', fontweight='bold', fontsize=12)
    ax.set_title(f'Final Performance Comparison (Round {clients_data[0]["num_rounds"]})', 
                 fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(client_names)
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrices(clients_data: List[Dict]):
    """Plot confusion matrices for each client."""
    
    n_clients = len(clients_data)
    fig, axes = plt.subplots(1, n_clients, figsize=(6*n_clients, 5))
    if n_clients == 1:
        axes = [axes]
    
    fig.suptitle('Confusion Matrices (Final Round)', fontsize=16, fontweight='bold')
    
    for idx, client in enumerate(clients_data):
        ax = axes[idx]
        cm = np.array(client['confusion_matrix'])
        
        # Plot heatmap
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, 
                   cbar_kws={'label': 'Count'}, square=True,
                   annot_kws={'size': 14, 'weight': 'bold'})
        
        ax.set_title(f"{client['client_name']}\nAccuracy: {client['final_accuracy']:.2f}% (+{client['improvement']:.1f}%)\n({client['num_rounds']} training rounds)", 
                    fontweight='bold', fontsize=12)
        ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=11)
        ax.set_ylabel('True Label', fontweight='bold', fontsize=11)
        ax.set_xticklabels(['Healthy (0)', 'ALL (1)'], fontsize=10)
        ax.set_yticklabels(['Healthy (0)', 'ALL (1)'], fontsize=10)
    
    plt.tight_layout()
    plt.show()

def print_summary(clients_data: List[Dict]):
    """Print summary statistics showing round-by-round progress."""
    print("="*80)
    print("📊 FL TRAINING SUMMARY (ROUND-BY-ROUND PROGRESS)")
    print("="*80)
    
    for client in clients_data:
        print(f"\n{client['client_name']} (Client ID: {client['client_id']}):")
        print(f"  Training Rounds: {client['num_rounds']}")
        print(f"\n  Round-by-Round Progress:")
        for round_data in client['rounds']:
            round_num = round_data['round']
            if round_num == 0:
                print(f"    Round {round_num} (Pre-FL):  Acc={round_data['accuracy']:.2f}%, Loss={round_data['loss']:.4f}")
            else:
                print(f"    Round {round_num}:           Acc={round_data['accuracy']:.2f}%, Loss={round_data['loss']:.4f}")
        
        print(f"\n  Overall Improvement:")
        print(f"    Pre-FL:  {client['pre_fl_accuracy']:.2f}%")
        print(f"    Final:   {client['final_accuracy']:.2f}%")
        print(f"    Change:  +{client['improvement']:.2f}%")
        
        print(f"\n  Final Metrics (Round {client['num_rounds']}):")
        print(f"    Loss:      {client['final_loss']:.4f}")
        print(f"    F1 Score:  {client['final_f1']:.2f}%")
        print(f"    Recall:    {client['final_recall']:.2f}%")
        print(f"    Precision: {client['final_precision']:.2f}%")
        
        cm = client['confusion_matrix']
        tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
        print(f"\n  Confusion Matrix:")
        print(f"    TN: {tn:4d}  FP: {fp:4d}")
        print(f"    FN: {fn:4d}  TP: {tp:4d}")
    
    print("\n" + "="*80 + "\n")

# ==========================================
# MAIN EXECUTION
# ==========================================

def main():
    print("="*80)
    print("🚀 FL ROUND-BY-ROUND METRICS VISUALIZATION")
    print("="*80 + "\n")
    
    try:
        # Connect and fetch
        supabase = connect_to_supabase()
        clients_data = fetch_metrics(supabase, SIMULATION_ID, CLIENT_ID)
        
        # Print summary
        print_summary(clients_data)
        
        # Generate visualizations
        print("📊 Generating visualizations...\n")
        
        # Individual client plots showing round-by-round progress
        plot_all_clients_metrics(clients_data)
        
        # Comparison plots (only if multiple clients)
        if len(clients_data) > 1:
            print("\n📊 Generating comparison plots...")
            plot_improvement(clients_data)
            plot_final_comparison(clients_data)
        
        # Confusion matrices
        plot_confusion_matrices(clients_data)
        
        print("\n✅ All visualizations generated successfully!")
        print(f"📊 Visualized {clients_data[0]['num_rounds']} training rounds per client")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

# Run
main()